# Explore-Exploit Tradeoff: Interactive Lab

**Goal:** Compare pure exploration, pure exploitation, and balanced strategies using interactive controls.

**Time:** 15 minutes

In this notebook, you'll create a simple 3-arm bandit environment representing three commodity sectors, then watch how different exploration strategies perform. You'll use an interactive slider to see how the epsilon parameter affects the explore-exploit balance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## The Setup: Three Commodity Sectors

You're allocating weekly trading capital across three commodity sectors:

- **Energy (crude oil, nat gas):** Mean return = 0.15% per week, std = 2.5%
- **Metals (gold, copper):** Mean return = 0.25% per week, std = 1.8%
- **Agriculture (wheat, corn):** Mean return = 0.18% per week, std = 2.2%

**Unknown to you:** Metals has the highest expected return.

**Your task:** Figure out which sector to focus on while still earning money.

In [ ]:
# Three-arm bandit: commodity sectors
arm_names = ['Energy', 'Metals', 'Agriculture']
arm_means = np.array([0.15, 0.25, 0.18])  # Mean returns (unknown to algorithm)
arm_stds = np.array([2.5, 1.8, 2.2])      # Return volatility

n_arms = len(arm_means)
best_arm = np.argmax(arm_means)

print("TRUE ARM VALUES (unknown to the algorithm):")
for i, name in enumerate(arm_names):
    marker = ' ⭐ BEST' if i == best_arm else ''
    print(f"  {name:12s}: μ = {arm_means[i]:5.2f}%, σ = {arm_stds[i]:.1f}%{marker}")

print(f"\nYour job: Discover that {arm_names[best_arm]} is best while minimizing regret.")

## Strategy 1: Pure Exploration (Random)

**Policy:** Pick a random arm every time.

**Why it fails:** Never exploits what you learn. Always wastes 2/3 of allocations on non-best arms.

In [ ]:
def pure_exploration(n_rounds=1000):
    """Random arm selection every round."""
    rewards = []
    regrets = []
    arm_counts = np.zeros(n_arms)
    
    for t in range(n_rounds):
        # Pick random arm
        arm = np.random.randint(n_arms)
        
        # Observe reward (normally distributed)
        reward = np.random.normal(arm_means[arm], arm_stds[arm])
        rewards.append(reward)
        arm_counts[arm] += 1
        
        # Calculate regret
        instant_regret = arm_means[best_arm] - arm_means[arm]
        regrets.append(instant_regret)
    
    return np.array(rewards), np.array(regrets), arm_counts

rewards_explore, regrets_explore, counts_explore = pure_exploration(1000)

print("Pure Exploration Results:")
print(f"  Total reward:  {np.sum(rewards_explore):7.2f}%")
print(f"  Avg reward:    {np.mean(rewards_explore):7.2f}% per week")
print(f"  Total regret:  {np.sum(regrets_explore):7.2f}%")
print(f"\n  Arm selection:")
for i, name in enumerate(arm_names):
    print(f"    {name:12s}: {counts_explore[i]:4.0f} times ({counts_explore[i]/10:.1f}%)")

## Strategy 2: Pure Exploitation (Greedy)

**Policy:** Always pick the arm with the highest observed average reward.

**Why it fails:** If initial samples are unlucky, you lock into the wrong arm forever.

In [ ]:
def pure_exploitation(n_rounds=1000, initial_explore=10):
    """Greedy selection after initial exploration."""
    Q = np.zeros(n_arms)  # Estimated values
    N = np.zeros(n_arms)  # Visit counts
    rewards = []
    regrets = []
    arm_counts = np.zeros(n_arms)
    
    for t in range(n_rounds):
        # Initial exploration: try each arm a few times
        if t < initial_explore * n_arms:
            arm = t % n_arms
        else:
            # Pure exploitation: pick best estimate
            arm = np.argmax(Q)
        
        # Observe reward
        reward = np.random.normal(arm_means[arm], arm_stds[arm])
        rewards.append(reward)
        arm_counts[arm] += 1
        
        # Update estimate (incremental average)
        N[arm] += 1
        Q[arm] += (reward - Q[arm]) / N[arm]
        
        # Calculate regret
        instant_regret = arm_means[best_arm] - arm_means[arm]
        regrets.append(instant_regret)
    
    return np.array(rewards), np.array(regrets), arm_counts, Q

rewards_exploit, regrets_exploit, counts_exploit, Q_final = pure_exploitation(1000)

print("Pure Exploitation Results:")
print(f"  Total reward:  {np.sum(rewards_exploit):7.2f}%")
print(f"  Avg reward:    {np.mean(rewards_exploit):7.2f}% per week")
print(f"  Total regret:  {np.sum(regrets_exploit):7.2f}%")
print(f"\n  Final value estimates:")
for i, name in enumerate(arm_names):
    print(f"    {name:12s}: Q = {Q_final[i]:5.2f}% (true μ = {arm_means[i]:.2f}%)")
print(f"\n  Arm selection:")
for i, name in enumerate(arm_names):
    print(f"    {name:12s}: {counts_exploit[i]:4.0f} times ({counts_exploit[i]/10:.1f}%)")

## Strategy 3: Epsilon-Greedy (Balanced)

**Policy:** With probability ε, explore (pick random arm). With probability 1-ε, exploit (pick best estimate).

**Why it works:** Balances learning and earning. Never fully commits to a potentially wrong estimate.

In [ ]:
def epsilon_greedy(n_rounds=1000, epsilon=0.1):
    """Epsilon-greedy: explore with probability epsilon."""
    Q = np.zeros(n_arms)
    N = np.zeros(n_arms)
    rewards = []
    regrets = []
    arm_counts = np.zeros(n_arms)
    exploration_count = 0
    
    for t in range(n_rounds):
        # Epsilon-greedy policy
        if np.random.rand() < epsilon:
            arm = np.random.randint(n_arms)  # Explore
            exploration_count += 1
        else:
            arm = np.argmax(Q)  # Exploit
        
        # Observe reward
        reward = np.random.normal(arm_means[arm], arm_stds[arm])
        rewards.append(reward)
        arm_counts[arm] += 1
        
        # Update estimate
        N[arm] += 1
        Q[arm] += (reward - Q[arm]) / N[arm]
        
        # Calculate regret
        instant_regret = arm_means[best_arm] - arm_means[arm]
        regrets.append(instant_regret)
    
    return np.array(rewards), np.array(regrets), arm_counts, Q, exploration_count

rewards_eps, regrets_eps, counts_eps, Q_eps, n_explore = epsilon_greedy(1000, epsilon=0.1)

print("Epsilon-Greedy Results (ε = 0.1):")
print(f"  Total reward:  {np.sum(rewards_eps):7.2f}%")
print(f"  Avg reward:    {np.mean(rewards_eps):7.2f}% per week")
print(f"  Total regret:  {np.sum(regrets_eps):7.2f}%")
print(f"\n  Exploration: {n_explore} / 1000 rounds ({n_explore/10:.1f}%)")
print(f"\n  Final value estimates:")
for i, name in enumerate(arm_names):
    print(f"    {name:12s}: Q = {Q_eps[i]:5.2f}% (true μ = {arm_means[i]:.2f}%)")
print(f"\n  Arm selection:")
for i, name in enumerate(arm_names):
    print(f"    {name:12s}: {counts_eps[i]:4.0f} times ({counts_eps[i]/10:.1f}%)")

## Visual Comparison

Let's compare all three strategies side-by-side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Cumulative reward
axes[0].plot(np.cumsum(rewards_explore), label='Pure Exploration', linewidth=2, alpha=0.7)
axes[0].plot(np.cumsum(rewards_exploit), label='Pure Exploitation', linewidth=2, alpha=0.7)
axes[0].plot(np.cumsum(rewards_eps), label='Epsilon-Greedy (ε=0.1)', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Week', fontsize=12)
axes[0].set_ylabel('Cumulative Return (%)', fontsize=12)
axes[0].set_title('Total Earnings Over Time', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Cumulative regret
axes[1].plot(np.cumsum(regrets_explore), label='Pure Exploration', linewidth=2, alpha=0.7)
axes[1].plot(np.cumsum(regrets_exploit), label='Pure Exploitation', linewidth=2, alpha=0.7)
axes[1].plot(np.cumsum(regrets_eps), label='Epsilon-Greedy (ε=0.1)', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Week', fontsize=12)
axes[1].set_ylabel('Cumulative Regret (%)', fontsize=12)
axes[1].set_title('Opportunity Cost Over Time', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print(f"  - Pure exploration: linear regret (never stops wasting on bad arms)")
print(f"  - Pure exploitation: depends on luck (might lock into wrong arm)")
print(f"  - Epsilon-greedy: best of both worlds (learns quickly, exploits mostly)")

## Interactive Widget: Tune the Epsilon Parameter

Use the slider to see how different epsilon values affect performance.

- **Low ε (0.01):** Mostly exploit, little exploration
- **Medium ε (0.1):** Balanced
- **High ε (0.3):** Lots of exploration, less exploitation

In [ ]:
def run_epsilon_comparison(epsilon=0.1, n_rounds=1000):
    """Run epsilon-greedy with interactive epsilon control."""
    np.random.seed(42)
    
    rewards, regrets, counts, Q, n_explore = epsilon_greedy(n_rounds, epsilon)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Plot 1: Cumulative reward
    axes[0].plot(np.cumsum(rewards), linewidth=2, color='green')
    axes[0].set_xlabel('Week')
    axes[0].set_ylabel('Cumulative Return (%)')
    axes[0].set_title(f'Total Earnings (ε={epsilon:.2f})')
    axes[0].grid(alpha=0.3)
    
    # Plot 2: Cumulative regret
    axes[1].plot(np.cumsum(regrets), linewidth=2, color='crimson')
    axes[1].set_xlabel('Week')
    axes[1].set_ylabel('Cumulative Regret (%)')
    axes[1].set_title(f'Opportunity Cost (ε={epsilon:.2f})')
    axes[1].grid(alpha=0.3)
    
    # Plot 3: Arm selection distribution
    axes[2].bar(arm_names, counts, color=['steelblue', 'gold', 'green'], alpha=0.7)
    axes[2].set_ylabel('Number of Selections')
    axes[2].set_title('Arm Selection Distribution')
    axes[2].axhline(y=counts[best_arm], color='red', linestyle='--', 
                    linewidth=1, label=f'Best arm ({arm_names[best_arm]})')
    axes[2].legend()
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nResults for ε = {epsilon:.2f}:")
    print(f"  Total reward:    {np.sum(rewards):7.2f}%")
    print(f"  Total regret:    {np.sum(regrets):7.2f}%")
    print(f"  Exploration %:   {n_explore/n_rounds*100:5.1f}%")
    print(f"  Best arm picks:  {counts[best_arm]:4.0f} / {n_rounds} ({counts[best_arm]/n_rounds*100:.1f}%)")

# Create interactive widget
interact(run_epsilon_comparison, 
         epsilon=FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1, description='Epsilon:'),
         n_rounds=IntSlider(min=100, max=2000, step=100, value=1000, description='Rounds:'));

## What Happens When the Best Arm Changes?

Real markets aren't stationary — the best commodity sector changes with economic regimes.

**Scenario:** Metals is best for first 500 weeks, then Energy becomes best (market shift).

In [ ]:
def non_stationary_bandit(n_rounds=1000, epsilon=0.1, shift_at=500):
    """Bandit where best arm changes mid-way."""
    # Initial means: Metals best
    means_phase1 = np.array([0.15, 0.25, 0.18])
    # After shift: Energy best (regime change, e.g., oil shock)
    means_phase2 = np.array([0.30, 0.20, 0.18])
    
    Q = np.zeros(n_arms)
    N = np.zeros(n_arms)
    rewards = []
    regrets = []
    chosen_arms = []
    
    for t in range(n_rounds):
        # Determine current means
        current_means = means_phase1 if t < shift_at else means_phase2
        current_best = np.argmax(current_means)
        
        # Epsilon-greedy policy
        if np.random.rand() < epsilon:
            arm = np.random.randint(n_arms)
        else:
            arm = np.argmax(Q)
        
        chosen_arms.append(arm)
        
        # Observe reward from current regime
        reward = np.random.normal(current_means[arm], arm_stds[arm])
        rewards.append(reward)
        
        # Update estimate
        N[arm] += 1
        Q[arm] += (reward - Q[arm]) / N[arm]
        
        # Calculate regret
        instant_regret = current_means[current_best] - current_means[arm]
        regrets.append(instant_regret)
    
    return np.array(rewards), np.array(regrets), np.array(chosen_arms)

rewards_shift, regrets_shift, arms_shift = non_stationary_bandit(1000, epsilon=0.1, shift_at=500)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Arm selection over time
colors = ['steelblue', 'gold', 'green']
for arm_idx in range(n_arms):
    arm_selected = (arms_shift == arm_idx).astype(int)
    # Smooth with rolling window
    window = 50
    smoothed = np.convolve(arm_selected, np.ones(window)/window, mode='valid')
    axes[0].plot(range(len(smoothed)), smoothed, label=arm_names[arm_idx], 
                linewidth=2, color=colors[arm_idx])

axes[0].axvline(x=500-25, color='red', linestyle='--', linewidth=2, label='Regime Shift')
axes[0].set_xlabel('Week', fontsize=12)
axes[0].set_ylabel('Selection Probability (50-week rolling)', fontsize=12)
axes[0].set_title('Arm Selection Over Time (Non-Stationary)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Cumulative regret
axes[1].plot(np.cumsum(regrets_shift), linewidth=2, color='crimson')
axes[1].axvline(x=500, color='red', linestyle='--', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Week', fontsize=12)
axes[1].set_ylabel('Cumulative Regret (%)', fontsize=12)
axes[1].set_title('Regret Accumulation (Notice the Kink)', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔄 Non-Stationary Behavior:")
print(f"  - Before week 500: Algorithm learned to prefer Metals (correct)")
print(f"  - After week 500: Energy became best (regime shift)")
print(f"  - Epsilon exploration allowed the algorithm to detect the shift")
print(f"  - Regret growth rate changes at the shift point (see the kink)")
print(f"\n  This is why pure exploitation fails in real markets!")

## Summary

**What you learned:**

1. **Pure exploration** wastes opportunities by never exploiting what you learn
2. **Pure exploitation** risks locking into suboptimal choices based on early luck
3. **Epsilon-greedy** balances both: mostly exploit, occasionally explore
4. The epsilon parameter controls the tradeoff: lower ε = more exploitation, higher ε = more exploration
5. In **non-stationary environments** (real markets), continued exploration is essential to detect regime changes

**Key insight:**
- Small ε (1-5%): Good for stable environments, long horizons
- Medium ε (10-15%): Balanced for typical trading scenarios
- Large ε (20-30%): Good for highly uncertain or rapidly changing environments

**What's next:**
- **Notebook 03:** Apply these concepts to real commodity price data from Yahoo Finance
- **Module 1:** Deep dive into epsilon-greedy variants (fixed, decaying, adaptive)
- **Module 2:** UCB algorithm — exploration based on confidence bounds, not random chance

**Try this:**
- Go back to the interactive widget and try ε = 0.01, 0.05, 0.2, 0.4
- Notice how low epsilon gets stuck if unlucky early, high epsilon wastes too much on exploration
- For the non-stationary scenario, try different epsilon values — does higher ε adapt faster after the shift?